# CardioScope-XAI — Exploratory Data Analysis
**Dataset:** UCI Cleveland Heart Disease (303 patients, 13 clinical features)

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_and_clean, FEATURE_COLUMNS, CATEGORICAL_FEATURES, CONTINUOUS_FEATURES

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Load Data

In [ ]:
df = load_and_clean('../data/raw/heart_disease_uci.csv')
df.head()

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nDtypes:\n{df.dtypes}')
print(f'\nMissing values:\n{df.isnull().sum()}')

In [ ]:
df.describe().T.round(2)

## 2. Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = df['target'].value_counts()
labels = ['No Disease (0)', 'Disease (1)']

axes[0].bar(labels, counts.values, color=['#4CAF50', '#F44336'], width=0.5)
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=90)
axes[1].set_title('Class Proportions')

plt.suptitle('Target Variable Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Class imbalance ratio: {counts[0]/counts[1]:.2f}')

## 3. Continuous Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

cont_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
feature_labels = {
    'age': 'Age (years)',
    'trestbps': 'Resting BP (mm Hg)',
    'chol': 'Cholesterol (mg/dl)',
    'thalach': 'Max Heart Rate',
    'oldpeak': 'ST Depression',
    'ca': 'Major Vessels (Ca)'
}

for i, feat in enumerate(cont_features):
    for target_val, color, label in [(0, '#4CAF50', 'No Disease'), (1, '#F44336', 'Disease')]:
        axes[i].hist(df[df['target'] == target_val][feat], bins=20,
                     alpha=0.6, color=color, label=label, edgecolor='white')
    axes[i].set_title(feature_labels[feat])
    axes[i].legend(fontsize=8)
    axes[i].set_ylabel('Count')

plt.suptitle('Continuous Feature Distributions by Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Categorical Feature Analysis

In [ ]:
cat_features = {
    'cp':      ['Typical Angina', 'Atypical Angina', 'Non-Anginal', 'Asymptomatic'],
    'restecg': ['Normal', 'ST-T Abnormality', 'LV Hypertrophy'],
    'slope':   ['Upsloping', 'Flat', 'Downsloping'],
    'thal':    ['Normal (3)', 'Fixed Defect (6)', 'Reversable Defect (7)'],
    'sex':     ['Female', 'Male'],
    'fbs':     ['FBS ≤120', 'FBS >120'],
    'exang':   ['No Angina', 'Exercise Angina']
}

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, (feat, _) in enumerate(cat_features.items()):
    ct = pd.crosstab(df[feat], df['target'])
    ct.plot(kind='bar', ax=axes[i], color=['#4CAF50', '#F44336'],
            edgecolor='white', alpha=0.85)
    axes[i].set_title(feat.upper())
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Count')
    axes[i].legend(['No Disease', 'Disease'], fontsize=8)
    axes[i].tick_params(axis='x', rotation=0)

axes[-1].set_visible(False)
plt.suptitle('Categorical Features vs Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(13, 10))

corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, square=True, linewidths=0.5, ax=ax,
    annot_kws={'size': 8}, vmin=-1, vmax=1
)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## 6. Feature Correlation with Target

In [ ]:
target_corr = df.corr()['target'].drop('target').sort_values()

colors = ['#F44336' if v > 0 else '#4CAF50' for v in target_corr.values]

fig, ax = plt.subplots(figsize=(8, 7))
bars = ax.barh(target_corr.index, target_corr.values, color=colors, edgecolor='white', height=0.6)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Pearson Correlation with Target')
ax.set_title('Feature Correlation with Heart Disease', fontsize=13, fontweight='bold')

for bar, val in zip(bars, target_corr.values):
    ax.text(val + (0.01 if val >= 0 else -0.01), bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.show()

print('Top 5 risk-increasing features:')
print(target_corr.tail(5).to_string())
print('\nTop 5 risk-reducing features:')
print(target_corr.head(5).to_string())

## 7. Boxplots — Continuous Features by Target

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 5))

for i, feat in enumerate(['age', 'trestbps', 'chol', 'thalach', 'oldpeak']):
    sns.boxplot(
        data=df, x='target', y=feat, ax=axes[i],
        palette={0: '#4CAF50', 1: '#F44336'}
    )
    axes[i].set_xticklabels(['No Disease', 'Disease'])
    axes[i].set_title(feat)
    axes[i].set_xlabel('')

plt.suptitle('Continuous Features by Heart Disease Status', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Key EDA Findings

| Finding | Implication |
|---|---|
| `thalach` (max HR) negatively correlated with disease | Lower max HR → higher risk |
| `cp` type 4 (asymptomatic) strongly associated with disease | Counter-intuitive — silent ischemia |
| `oldpeak` (ST depression) positively correlated | Key exercise stress marker |
| `ca` (vessels colored) positively correlated | More blocked vessels → higher risk |
| `exang` (exercise angina) positively correlated | Pain during exercise = risk signal |
| Class balance is ~54% No Disease / 46% Disease | Relatively balanced — no heavy resampling needed |